In [1]:
# Author: Niko Bleidistel
# last change: 2026-08-10

# Package Import

In [2]:
from pathlib import Path 
from os import makedirs
import sys
import importlib

import re

import pandas as pd
import numpy as np
import math

from typing import cast
import matplotlib as mpl
import matplotlib.pyplot as plt

In [3]:
PYTHON_HELPER_FOLDER = Path(r"py-helpers")

# Add the path to the custom packages to sys.path so that they can be imported
sys.path.append(str(PYTHON_HELPER_FOLDER.resolve()))

# import custom packages
import comsol_data_export as cde
import time_logging as tl

import comsol_data_plotting as cdp
import plot_functions as pfs

# reload custom packages (for each execution) to reflect recent changes
_ = importlib.reload(cde)
_ = importlib.reload(tl)

_ = importlib.reload(cdp)
_ = importlib.reload(pfs)

# PATHS

In [4]:
MAIN_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\10_bachelor_thesis_models_use_terminals")
INPUT_FOLDER = MAIN_FOLDER / "Test Output"
OUTPUT_FOLDER = MAIN_FOLDER / "PLOTS"

makedirs(OUTPUT_FOLDER, exist_ok=True)  # create output folder if it doesn't exist

In [5]:
if False:
    input_folder = INPUT_FOLDER

    def mask(filename):
        return not (str(filename).endswith('.mph') or str(filename).endswith('.csv'))

    import seedir as sd
    sd.seedir(input_folder, style='lines', mask=mask)

# INITIALIZE

In [6]:
# initialize time logging
_ = tl.initialize_time_log(OUTPUT_FOLDER / 'time_log.csv')

## data folders

In [7]:
folders = [f for f in INPUT_FOLDER.rglob("*") if f.is_dir()]
data_export_folders = [f for f in folders if "Data Export" in f.name]
sweep_export_folders = [f for f in folders if "Sweep Export" in f.name]

if True:
    display(data_export_folders)
    display(sweep_export_folders)

[WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/02_00-H Designs/02_00_a-H design with minimized insulator/Data Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/02_00-H Designs/02_00_b-H design with full insulator/Data Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/01_xx-Coils and Grid Designs/01_01_a-Round spiral/Data Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/01_xx-Coils and Grid Designs/01_01_b-Rectangular spiral/Data Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/01_xx-Coils and Grid Designs/01_02_a-Grid/Data Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/01_xx-Coils and Grid Designs/01_03_a

[WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/02_00-H Designs/02_00_b-H design with full insulator/Sweep/Sweep Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/02_00-H Designs/02_00_a-H design with minimized insulator/Sweep/Sweep Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/01_xx-Coils and Grid Designs/01_03_b-Rectangular spiral combined with grid/Sweep - Voltage angles/Sweep Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/01_xx-Coils and Grid Designs/01_03_a-Round spiral combined with grid/Sweep - Voltage angles/Sweep Export'),
 WindowsPath('R:/Bleidistel_Niko/COMSOL/COMSOL Files/10_bachelor_thesis_models_use_terminals/Test Output/01_xx-Coils and Grid Designs/01_02_a-Grid/Sweep - Voltage angles and left_out_lines/left_out_lines

# TRANSLATION (Constants)

In [8]:
TRANSLATE_PLOTLABELS = {
    "x":                r"$x$-axis [m]",
    "y":                r"$y$-axis [m]",
    "z":                r"$z$-axis [m]",
    "mf.normB (T)":     r"Magnetic flux density, norm $|\vec{B}|$ [T]",
    "mf.Bx (T)":        r"Magnetic flux density, x-component $B_x$ [T]", 
    "mf.By (T)":        r"Magnetic flux density, y-component $B_y$ [T]", 
    "mf.Bz (T)":        r"Magnetic flux density, z-component $B_z$ [T]",
    "T (K)":            r"Temperature $T$ [K]",
    "V (V)":            r"Electric potential $V$ [V]",
    "ec.normJ (A/m^2)": r"Current density, norm $|\vec{J}|$ [A/m^2]",
    "ec.Jx (A/m^2)":    r"Current density, x-component $J_x$ [A/m^2]",
    "ec.Jy (A/m^2)":    r"Current density, y-component $J_y$ [A/m^2]",
    "ec.Jz (A/m^2)":    r"Current density, z-component $J_z$ [A/m^2]",
    "Set angle (°)":    r"Set angle $\theta$ [°]",
    "Angle Error (°)":  r"Angle error $\Delta\theta$ [°]"
}

In [9]:
X_AXIS_PARAMS = ["x", "y", "z"]
Y_AXIS_PARAMS = [key for key in TRANSLATE_PLOTLABELS.keys() if key not in X_AXIS_PARAMS]

# STANDARD PLOTS

In [10]:
if False:
    output_folder = OUTPUT_FOLDER / "Standard Plots"

    for data_export_folder in data_export_folders:
        tl.log_message(f"Starting creating standard plots for model: {data_export_folder.parent.stem}")

        # define output folder for plots
        plot_folder = output_folder / data_export_folder.parent.stem

        # get a list of all csv files in the "Data Export" subfolder
        tl.log_message(f"Importing data from CSV files in folder: {data_export_folder}")
        df_terminal = pd.DataFrame()
        df_parameters = pd.DataFrame()

        csv_files = list(data_export_folder.glob("*.csv"))
        for csv_file in csv_files:
            if re.match(r".*-terminals$", csv_file.stem):
                df_terminal = pd.read_csv(csv_file)
                tl.log_message(f"Imported terminal data")
            elif re.match(r".*-parameters$", csv_file.stem):
                df_parameters = pd.read_csv(csv_file)
                tl.log_message(f"Imported parameter data")


        # get a list of all txt files in the "Data Export" subfolder
        txt_files = list(data_export_folder.glob("*.txt"))
        for txt_file in txt_files:
            tl.log_message(f"Processing txt file: {txt_file.stem}")
            header_data, df = cdp.read_comsol_export(str(txt_file))
            modelstem = str(header_data.get('Model')).replace(".mph", "")

            title_match = re.match(r"(.*)-(.*)-.*", modelstem)
            modeltitle = title_match.group(2) if title_match else modelstem



            if re.match(r".*-depth_exported_data$", txt_file.stem):
                df.drop(labels=["x", "y"], axis=1, inplace=True)

                depth_folder = plot_folder / "z axis"
                makedirs(depth_folder, exist_ok=True)  # create output folder if it doesn't exist

                xparam = "z"
                for yparam in Y_AXIS_PARAMS:
                    if yparam in df.columns:
                        tl.log_message(f"Started plotting of {yparam} vs {xparam}")
                        fig, ax = cdp.standard_plot(
                                        output_folder = str(depth_folder),
                                        x_param = xparam,
                                        y_param = yparam,
                                        header_data = header_data,
                                        df_curves = df,
                                        df_param = df_parameters,
                                        title = modeltitle,
                                        add_title_info = False,
                                        translation_dict = TRANSLATE_PLOTLABELS,
                                        color = 'midnightblue',
                                        save_plot = True,
                                        )
                        tl.log_message(f"Finished plotting of {yparam} vs {xparam}")



            elif re.match(r".*-homogeneity_exported_data$", txt_file.stem):
                df.drop(labels=["x"], axis=1, inplace=True)

                # create a copy of the original DataFrame and round the "z" values to 6 significant digits to avoid floating point precision issues  
                df["z"] = cde.round_to_6_sig_digits(df["z"])
                unique_z_values = df["z"].dropna().unique()

                cmap = mpl.colormaps['viridis'] # type: ignore

                xparam = "y"
                homogeneity_folder = plot_folder / "y axis"
                makedirs(homogeneity_folder, exist_ok=True)  # create output folder if it doesn't exist

                for yparam in Y_AXIS_PARAMS:
                    if yparam in df.columns:
                        tl.log_message(f"Started plotting of {yparam} vs {xparam}")

                        fig, ax = None, None  
                        for loop_counter, z_value in enumerate(unique_z_values):
                            df_subset = df[df["z"] == z_value]
                            color = cmap(loop_counter / len(unique_z_values))  # get color from colormap based on index
        
                            fig, ax = cdp.standard_plot(
                                            output_folder = None,
                                            x_param = xparam,
                                            y_param = yparam,
                                            header_data = header_data,
                                            df_curves = df_subset,
                                            df_param = df_parameters,
                                            sweep_params = ["z"],
                                            title = modeltitle,
                                            add_title_info = False,
                                            translation_dict = TRANSLATE_PLOTLABELS,
                                            fig = fig, 
                                            ax = ax,
                                            color=color,
                                            save_plot = False,
                                            )
                            ax.legend(
                                loc="upper center", 
                                bbox_to_anchor=(0.5, -0.15), 
                                ncol=3
                                )
                            try:
                                xlength = df_parameters[df_parameters["name"] == "conductor_all_length"]["evaluated_value"].item()
                            except:
                                xlength = df_parameters[df_parameters["name"] == "conductor_A_length"]["evaluated_value"].item()
                            ax.set_xlim(left=-0.5*xlength, right=0.5*xlength)

                            
                        modelname = str(header_data.get('Model')).replace(".mph", "")
                        output_path = homogeneity_folder / f"{modelname}_{xparam}_vs_{yparam}.png"
                        fig.savefig(str(output_path), dpi=300, bbox_inches="tight") if fig is not None else None
                        tl.log_message(f"Finished plotting of {yparam} vs {xparam}")



            elif re.match(r".*-longitudinal_exported_data$", txt_file.stem):
                df.drop(labels=["y"], axis=1, inplace=True)

                # create a copy of the original DataFrame and round the "z" values to 6 significant digits to avoid floating point precision issues
                df = df.copy()
                df["z"] = cde.round_to_6_sig_digits(df["z"])
                unique_z_values = df["z"].dropna().unique()

                cmap = mpl.colormaps['viridis'] # type: ignore

                xparam = "x"
                longitudinal_folder = plot_folder / "x axis"
                makedirs(longitudinal_folder, exist_ok=True)  # create output folder if it doesn't exist

                for yparam in Y_AXIS_PARAMS:
                    if yparam in df.columns:
                        tl.log_message(f"Started plotting of {yparam} vs {xparam}")

                        fig, ax = None, None  
                        for loop_counter, z_value in enumerate(unique_z_values):
                            df_subset = df[df["z"] == z_value]
                            color = cmap(loop_counter / len(unique_z_values))  # get color from colormap based on index
                            
                            fig, ax = cdp.standard_plot(
                                            output_folder = None,
                                            x_param = xparam,
                                            y_param = yparam,
                                            header_data = header_data,
                                            df_curves = df_subset,
                                            df_param = df_parameters,
                                            sweep_params = ["z"],
                                            title = modeltitle,
                                            add_title_info = False,
                                            translation_dict = TRANSLATE_PLOTLABELS,
                                            fig = fig, 
                                            ax = ax,
                                            color=color,
                                            save_plot = False,
                                            )
                            ax.legend(
                                loc="upper center", 
                                bbox_to_anchor=(0.5, -0.15), 
                                ncol=3
                                )
                            try:
                                xlength = df_parameters[df_parameters["name"] == "conductor_all_length"]["evaluated_value"].item()
                            except:
                                xlength = df_parameters[df_parameters["name"] == "conductor_A_length"]["evaluated_value"].item()
                            ax.set_xlim(left=-0.5*xlength, right=0.5*xlength)

                            
                        modelname = str(header_data.get('Model')).replace(".mph", "")
                        output_path = longitudinal_folder / f"{modelname}_{xparam}_vs_{yparam}.png"
                        fig.savefig(str(output_path), dpi=300, bbox_inches="tight") if fig is not None else None
                        tl.log_message(f"Finished plotting of {yparam} vs {xparam}")



            elif re.match(r".*-conductor_exported_data$", txt_file.stem):
                df.drop(labels=["z"], axis=1, inplace=True)
                limit = 500e-6
                df = df[df["x"].between(-limit, limit, inclusive="neither") & df["y"].between(-limit, limit, inclusive="neither")]  # filter out points outside the range of -limit to limit for both x and y
                df = df.dropna()

                conductor_folder = plot_folder / "conductor plane"
                makedirs(conductor_folder, exist_ok=True)  # create output folder if it doesn't exist

                for zparam in Y_AXIS_PARAMS:
                    if zparam in df.columns:
                        tl.log_message(f"Started plotting of {zparam} over xy-plane in conductor")
                        fig, ax = cdp.plane_plot(
                            output_folder = str(conductor_folder),
                            x_param = "x",
                            y_param = "y",
                            z_param = zparam,
                            header_data = header_data,
                            df = df,
                            df_param = df_parameters,
                            title_params = None,
                            sweep_params = None,
                            title = modeltitle,
                            add_title_info = False,
                            custom_label= None,
                            translation_dict = TRANSLATE_PLOTLABELS,
                            save_plot = True,
                            fig = None, 
                            ax = None,
                            colormap='viridis',
                            labelcolor = 'black',
                            xscale = None,
                            yscale = None,
                            xstyle = 'prefix',
                            ystyle = 'prefix',
                            zstyle = 'prefix',
                            grid = True,
                            )
                        tl.log_message(f"Finished plotting of {zparam} over xy-plane in conductor")



            elif re.match(r".*-xy_exported_data$", txt_file.stem):
                df.drop(labels=["z"], axis=1, inplace=True)
                limit = 100e-6
                df = df[df["x"].between(-limit, limit, inclusive="neither") & df["y"].between(-limit, limit, inclusive="neither")]  # filter out points outside the range of -limit to limit for both x and y
                df = df.dropna()

                xy_plane_folder = plot_folder / "xy plane"
                makedirs(xy_plane_folder, exist_ok=True)  # create output folder if it doesn't exist

                for zparam in Y_AXIS_PARAMS:
                    if zparam in df.columns:
                        tl.log_message(f"Started plotting of {zparam} over xy-plane under conductor")
                        fig, ax = cdp.plane_plot(
                            output_folder = str(xy_plane_folder),
                            x_param = "x",
                            y_param = "y",
                            z_param = zparam,
                            header_data = header_data,
                            df = df,
                            df_param = df_parameters,
                            title_params = None,
                            sweep_params = None,
                            title = modeltitle,
                            add_title_info = False,
                            custom_label= None,
                            translation_dict = TRANSLATE_PLOTLABELS,
                            save_plot = True,
                            fig = None, 
                            ax = None,
                            colormap='viridis',
                            labelcolor = 'black',
                            xscale = None,
                            yscale = None,
                            xstyle = 'prefix',
                            ystyle = 'prefix',
                            zstyle = 'prefix',
                            grid = True,
                            )
                        tl.log_message(f"Finished plotting of {zparam} over xy-plane under conductor")



            else:
                tl.log_message(f"Skipped plotting for file: '{txt_file.stem}' (no standard plot type matched).")

                        
            plt.close('all')

# ERROR PLOT

In [11]:
ERROR_SWEEP_MODELS = [
    "01_02_a-Grid",
]

In [12]:
ERROR_DEPTH = -3e-6 #m
ANGLES = list(range(0, 91, 3))  # angles from 0 to 90 degrees in steps of 3 degrees

In [13]:
if True:
    output_folder = OUTPUT_FOLDER / "Angle Error Plots"
    makedirs(output_folder, exist_ok=True)  # create output folder if it doesn't exist
    
    for sweep_export_folder in sweep_export_folders:
        if "failed" in sweep_export_folder.parent.parent.stem:
            continue # skip failed sweeps

        if True: # whether to include the left_out_lines folder layer
            left_out_folder = sweep_export_folder.parent.stem
            plot_output_folder = output_folder / left_out_folder

            model_folder = sweep_export_folder.parent.parent.parent.stem
        else:
            plot_output_folder = output_folder
            model_folder = sweep_export_folder.parent.parent.stem

        if model_folder not in ERROR_SWEEP_MODELS:
            continue

        tl.log_message(f"Starting creating angle error plots for sweep of model: {model_folder}")

        title_match = re.match(r"(.*)-(.*)", model_folder)
        modeltitle = title_match.group(2) if title_match else model_folder

        columns = ["mf.Bx (T)", "mf.By (T)"]

        df_angle = pd.DataFrame(columns=["Set angle (°)", "Actual angle (°)", "Angle Error (°)"]+columns)

        txt_files = list(sweep_export_folder.glob("*.txt"))

        for txt_file in txt_files:
            iteration_match = re.search(r"-iteration_(.*)-depth_exported_data$", txt_file.stem)
            if iteration_match:
                iteration = int(iteration_match.group(1))
                angle = ANGLES[iteration] if iteration is not None and iteration < len(ANGLES) else None
                df_angle.loc[iteration, "Set angle (°)"] = angle

                # # avoid issues with long paths on Windows
                # path_str = str(txt_file.resolve())
                # extended_path = "\\\\?\\UNC\\" + path_str[2:]
                extended_path=txt_file

                header_data, df_txt = cdp.read_comsol_export(str(extended_path))

                df_txt = df_txt[["z"] + columns]
                df_txt = df_txt.dropna()

                for col in columns:
                    value = np.interp(ERROR_DEPTH, df_txt["z"], df_txt[col])
                    df_angle.loc[df_angle["Set angle (°)"] == angle, col] = value


                x = -1 * cast(float, df_angle.at[iteration, "mf.Bx (T)"]) # -1 is tempoary fix of old mistake
                y = -1 * cast(float, df_angle.at[iteration, "mf.By (T)"]) # -1 is tempoary fix of old mistake

                df_angle.loc[iteration, "Actual angle (°)"] = np.rad2deg(np.arctan2(y,x))
                df_angle.loc[iteration, "Angle Error (°)"] = cast(float, df_angle.at[iteration, "Actual angle (°)"])  - cast(float, df_angle.at[iteration, "Set angle (°)"])

        xparam = "Set angle (°)"
        yparam = "Angle Error (°)"
        tl.log_message(f"Started plotting of {yparam} vs {xparam}")

        # extract descriptions
        try: 
            x_discription = TRANSLATE_PLOTLABELS.get(xparam)
        except Exception as e:
            x_discription = xparam
    
        try:
            y_discription = TRANSLATE_PLOTLABELS.get(yparam)
        except Exception as e:
            y_discription = yparam

        unit = "m"
        siprefix, xsi_exponent, exponent_diff = pfs.get_SI_prefix((ERROR_DEPTH, ERROR_DEPTH))  # Get the SI prefix for the value
        scaled_value = ERROR_DEPTH * 10 ** (-xsi_exponent)

        
        # plot the data
        fig, ax = cdp.plot_comsol_data(
            df = df_angle,
            header_data = header_data,
            x_column = xparam,
            y_column = yparam,
            x_label = x_discription,
            y_label = y_discription,
            title = f"{modeltitle}: Angle Error at z = {scaled_value} [{siprefix}{unit}]",
            add_title_info = False,
            label = None,
            marker = 'o', 
            color = 'midnightblue', 
            labelcolor = 'black', 
            fig = None, 
            ax = None,
            show_legend = False,
            legend_loc = 'best',
            xscale = None,
            yscale = None,
            xstyle = 'plain',
            ystyle = 'plain',
            grid = True,
            )
        if True:
            modelname = str(header_data.get('Model')).replace(".mph", "")
            makedirs(plot_output_folder, exist_ok=True)  # create output folder if it

            output_path = plot_output_folder / f"{modelname}_{xparam}_vs_{yparam}.png"
            fig.savefig(str(output_path), dpi=300, bbox_inches="tight")

        fig, ax = cdp.polar_plot(
                    df = df_angle,
                    angle_column = xparam,
                    r_column = yparam,
                    angle_label = x_discription,
                    title = f"{modeltitle}: Angle Error at z = {scaled_value} [{siprefix}{unit}]",
                    label = None,
                    marker = 'o', 
                    color = 'midnightblue', 
                    labelcolor = 'black', 
                    fig = None, 
                    ax = None,
                    show_legend = False,
                    legend_loc = 'best',
                    grid = True,
                    )
        if True:
            modelname = str(header_data.get('Model')).replace(".mph", "")
            makedirs(plot_output_folder, exist_ok=True)  # create output folder if it
            
            output_path = plot_output_folder / f"{modelname}_{xparam}_vs_{yparam}_polar.png"
            fig.savefig(str(output_path), dpi=300, bbox_inches="tight")

        plt.close('all')
        tl.log_message(f"Finished plotting of {yparam} vs {xparam}")
